In [2]:
import pandas as pd
import numpy as np

# 1. Load all tables as text
acc = pd.read_csv('accounts.csv.gz', dtype=str)
txn = pd.read_csv('transactions.csv.gz', dtype=str)
sec = pd.read_csv('securities.csv.gz', dtype=str)
pos = pd.read_csv('positions.csv.gz', dtype=str)
cmp = pd.read_csv('comparison_positions.csv.gz', dtype=str)
prc = pd.read_csv('prices.csv.gz', dtype=str)
ca  = pd.read_csv('corporate_actions.csv.gz', dtype=str)
fx  = pd.read_csv('fx_rates.csv.gz', dtype=str)
ldg = pd.read_csv('ledger_nav.csv.gz', dtype=str)
pip = pd.read_csv('pipeline_events.csv.gz', dtype=str)

# --- A. MANIFEST & DUPLICATES (T01, T02) ---
print("1. MANIFEST COUNTS")
print(f"Transactions: {len(txn):,} rows | Unique IDs: {txn['transaction_id'].nunique():,}")
dup_ids = txn[txn.duplicated('transaction_id', keep=False)]
dup_id_count = txn['transaction_id'].nunique()
print(f"Repeated txn IDs: {len(txn) - dup_id_count:,} (Report: 2,105)")

cols_no_id = [c for c in txn.columns if c != 'transaction_id']
exact_dups = txn.duplicated(subset=txn.columns, keep='first').sum()
id_dups = txn.duplicated(subset=['transaction_id'], keep='first').sum()
print(f"Exact row duplicates: {exact_dups:,} (Report: 2,032) | Conflicting ID duplicates: {id_dups - exact_dups:,} (Report: 73)")

# --- B. ACCOUNTS & SECURITIES (M01-M06) ---
print("\n2. MASTER DATA")
m01 = acc['base_currency'].isna().sum()
# Valid ISO 3166-1 alpha-2 check (2 uppercase letters)
m02 = (~acc['domicile'].fillna('').str.match(r'^[A-Z]{2}$')).sum()
m03_isin_pairs = sec[sec['isin'].notna() & sec.duplicated('isin', keep=False)]['isin'].nunique()
m04_cusip_dups = sec[sec['cusip'].notna() & sec.duplicated('cusip', keep=False)]['cusip'].nunique()
m05_blank_cusip = sec['cusip'].isna().sum()
valid_fx_ccy = set(fx['currency'].dropna().unique())
m06_invalid_ccy = (~sec['currency'].isin(valid_fx_ccy)).sum()
print(f"M01 missing base_ccy: {m01} (400) | M02 invalid domicile: {m02} (300)")
print(f"M03 dup ISIN pairs: {m03_isin_pairs} (12) | M04 dup CUSIP: {m04_cusip_dups} | M05 blank CUSIP: {m05_blank_cusip} (18) | M06 invalid sec ccy: {m06_invalid_ccy} (12)")

# --- C. TRANSACTIONS (T03-T14) ---
print("\n3. TRANSACTIONS")
qty = pd.to_numeric(txn['quantity'], errors='coerce')
prc_num = pd.to_numeric(txn['price'], errors='coerce')
gross = pd.to_numeric(txn['gross_amount'], errors='coerce')

t03 = (qty.isna() | prc_num.isna() | txn['settlement_date'].isna()).sum()
t04 = (qty <= 0).sum()
t06 = ((gross - (qty * prc_num * 0.01)).abs() / (qty * prc_num * 0.01) < 0.01).sum() # 100x price check
print(f"T03 missing qty/price/settle: {t03:,} (6,609) | T04 qty <= 0: {t04:,} (655) | T06 100x price: {t06:,} (1,119)")

# Referential integrity & master alignment
txn_sec = txn.merge(sec[['security_id', 'cusip', 'currency']], on='security_id', how='left', suffixes=('', '_master'))
t11 = txn_sec['currency_master'].isna().sum()
t09 = (txn_sec['currency_master'].notna() & txn_sec['cusip_master'].notna() & (txn_sec['cusip'] != txn_sec['cusip_master'])).sum()
t10 = (txn_sec['currency_master'].notna() & (txn_sec['currency'] != txn_sec['currency_master'])).sum()
t12 = (~txn['account_id'].isin(set(acc['account_id']))).sum()
print(f"T09 CUSIP mismatch: {t09:,} (12,598) | T10 ccy mismatch: {t10:,} (3,547) | T11 unmastered sec: {t11:,} (1,131) | T12 unmastered acc: {t12:,} (706)")

# Orphans & double reversals
all_ids = set(txn['transaction_id'])
cancels = txn[txn['status'] == 'CANCELLED']
t07 = (~cancels['related_transaction_id'].isin(all_ids)).sum()
reversals = txn[txn['status'] == 'REVERSED']
rev_counts = reversals.groupby('related_transaction_id').size()
t08 = (rev_counts == 2).sum()
print(f"T07 orphan cancels: {t07:,} (842) | T08 double-reversed originals: {t08:,} (701)")

# --- D. POSITIONS & RECONCILIATION (R01-R03) ---
print("\n4. POSITIONS & RECONCILIATION")
pos['qty_num'] = pd.to_numeric(pos['quantity'], errors='coerce')
cmp['qty_num'] = pd.to_numeric(cmp['quantity'], errors='coerce')

recon = pos.merge(cmp, on=['account_id', 'security_id'], how='outer', suffixes=('_book', '_cust'))
in_both = recon[recon['qty_num_book'].notna() & recon['qty_num_cust'].notna()]
r01_diff = in_both[(in_both['qty_num_book'] - in_both['qty_num_cust']).abs() >= 1e-6]
r02_book_only = recon['qty_num_cust'].isna().sum()
r02_cust_only = recon['qty_num_book'].isna().sum()
print(f"In both: {len(in_both):,} (480,183) | R01 differing: {len(r01_diff):,} (1,917) | R02 book-only: {r02_book_only} (480) | cust-only: {r02_cust_only} (0)")

# Rebuild Interpretation B (Book replica: keep duplicates, BUY=+qty, SELL=-qty, REVERSED flips sign)
txn_calc = txn.copy()
txn_calc['qty_num'] = pd.to_numeric(txn_calc['quantity'], errors='coerce').fillna(0.0)
sign = np.where(txn_calc['transaction_type'] == 'BUY', 1.0, -1.0)
rev_flip = np.where(txn_calc['status'] == 'REVERSED', -1.0, 1.0)
txn_calc['signed_qty'] = txn_calc['qty_num'] * sign * rev_flip

hist_b = txn_calc.groupby(['account_id', 'security_id'], as_index=False)['signed_qty'].sum()
hist_b = hist_b[hist_b['signed_qty'].abs() >= 1e-6]

book_vs_hist = pos.merge(hist_b, on=['account_id', 'security_id'], how='inner')
r03_breaks = book_vs_hist[(book_vs_hist['qty_num'] - book_vs_hist['signed_qty']).abs() >= 0.01]
sign_inv = r03_breaks[(r03_breaks['qty_num'] + r03_breaks['signed_qty']).abs() < 0.01]
print(f"R03 book vs history breaks: {len(r03_breaks):,} (939) | Exact sign inversions: {len(sign_inv):,} (219)")

# --- E. CORPORATE ACTIONS, FX, LEDGER ---
print("\n5. CORPORATE ACTIONS, FX & LEDGER")
c01 = (ca['effective_date'] < ca['ex_date']).sum()
usd_fx = fx[fx['currency'] == 'USD'].copy()
usd_fx['rate_num'] = pd.to_numeric(usd_fx['rate_to_usd'], errors='coerce')
x01 = ((usd_fx['rate_num'] - 1.0).abs() > 1e-6).sum()

ldg['na'] = pd.to_numeric(ldg['net_assets'], errors='coerce')
ldg['sh'] = pd.to_numeric(ldg['shares_outstanding'], errors='coerce')
ldg['nav'] = pd.to_numeric(ldg['nav_per_share'], errors='coerce')
l01 = (((ldg['na'] / ldg['sh']) - ldg['nav']).abs() / ldg['nav'].abs() > 0.0001).sum()
l02 = (ldg['na'] < 0).sum()
print(f"C01 eff < ex: {c01} (87) | X01 USD != 1.0: {x01} (30) | L01 NAV > 1bp: {l01} (40) | L02 neg NA: {l02} (64)")

# --- F. THREE DEEP TRACES ---
print("\n6. TRACE VERIFICATION")
print("Trace A (TXN100002453):", len(txn[txn['transaction_id'] == 'TXN100002453']), "rows")
print("Trace A Pos (ACC1326787 / SEC689108):", pos[(pos['account_id']=='ACC1326787') & (pos['security_id']=='SEC689108')][['quantity', 'transaction_count']].to_dict('records'))
print("Trace B Pos (ACC6534055 / SEC619065):", pos[(pos['account_id']=='ACC6534055') & (pos['security_id']=='SEC619065')][['quantity', 'transaction_count']].to_dict('records'))
print("Trace C Pos (ACC1009764 / SEC115369):", pos[(pos['account_id']=='ACC1009764') & (pos['security_id']=='SEC115369')][['quantity', 'transaction_count']].to_dict('records'))

1. MANIFEST COUNTS
Transactions: 1,409,533 rows | Unique IDs: 1,407,428
Repeated txn IDs: 2,105 (Report: 2,105)
Exact row duplicates: 2,032 (Report: 2,032) | Conflicting ID duplicates: 73 (Report: 73)

2. MASTER DATA
M01 missing base_ccy: 400 (400) | M02 invalid domicile: 0 (300)
M03 dup ISIN pairs: 12 (12) | M04 dup CUSIP: 24 | M05 blank CUSIP: 18 (18) | M06 invalid sec ccy: 12 (12)

3. TRANSACTIONS
T03 missing qty/price/settle: 6,609 (6,609) | T04 qty <= 0: 655 (655) | T06 100x price: 1,772 (1,119)
T09 CUSIP mismatch: 8,352 (12,598) | T10 ccy mismatch: 3,547 (3,547) | T11 unmastered sec: 1,131 (1,131) | T12 unmastered acc: 706 (706)
T07 orphan cancels: 4,347 (842) | T08 double-reversed originals: 701 (701)

4. POSITIONS & RECONCILIATION
In both: 480,183 (480,183) | R01 differing: 1,918 (1,917) | R02 book-only: 480 (480) | cust-only: 0 (0)
R03 book vs history breaks: 5,237 (939) | Exact sign inversions: 218 (219)

5. CORPORATE ACTIONS, FX & LEDGER
C01 eff < ex: 87 (87) | X01 USD != 1.

In [4]:
import pandas as pd
import numpy as np

acc = pd.read_csv('accounts.csv.gz', dtype=str)
txn = pd.read_csv('transactions.csv.gz', dtype=str)
sec = pd.read_csv('securities.csv.gz', dtype=str)
pos = pd.read_csv('positions.csv.gz', dtype=str)
cmp = pd.read_csv('comparison_positions.csv.gz', dtype=str)

# 1. M02: Check non-ISO domicile codes
print("M02 domicile counts (bottom 10):")
print(acc['domicile'].value_counts(dropna=False).tail(10))

# 2. T06: 100x price on positive qty/price/gross
qty = pd.to_numeric(txn['quantity'], errors='coerce')
prc = pd.to_numeric(txn['price'], errors='coerce')
grs = pd.to_numeric(txn['gross_amount'], errors='coerce')
t06 = ((qty > 0) & (prc > 0) & (grs > 0) & (((prc / (grs / qty)) - 100).abs() < 1.0)).sum()
print(f"\nT06 100x price (qty>0): {t06:,} (Report: 1,119)")

# 3. T09: CUSIP mismatch including NaN transaction CUSIPs against non-null master CUSIPs
txn_sec = txn.merge(sec[['security_id', 'cusip', 'currency']], on='security_id', how='left', suffixes=('', '_master'))
t09 = (txn_sec['currency_master'].notna() & txn_sec['cusip_master'].notna() & (txn_sec['cusip'].fillna('') != txn_sec['cusip_master'])).sum()
print(f"T09 CUSIP mismatch (incl. blank txn CUSIP): {t09:,} (Report: 12,598)")

# 4. T07: Orphan cancels by transaction_type == 'CANCEL'
all_ids = set(txn['transaction_id'])
cancels_type = txn[txn['transaction_type'] == 'CANCEL']
t07 = (~cancels_type['related_transaction_id'].isin(all_ids)).sum()
print(f"T07 orphan cancels (type=='CANCEL'): {t07:,} (Report: 842)")

# 5. R01: Differing positions at 0.01 tolerance
pos['qty_num'] = pd.to_numeric(pos['quantity'], errors='coerce')
cmp['qty_num'] = pd.to_numeric(cmp['quantity'], errors='coerce')
recon = pos.merge(cmp, on=['account_id', 'security_id'], how='inner', suffixes=('_book', '_cust'))
r01 = ((recon['qty_num_book'] - recon['qty_num_cust']).abs() >= 0.01).sum()
print(f"R01 differing (tol >= 0.01): {r01:,} (Report: 1,917)")

# 6. R03: Book replica excluding CANCELLED rows
active_txn = txn[txn['status'] != 'CANCELLED'].copy()
active_txn['qty_num'] = pd.to_numeric(active_txn['quantity'], errors='coerce').fillna(0.0)
sign = np.where(active_txn['transaction_type'] == 'BUY', 1.0, -1.0)
rev_flip = np.where(active_txn['status'] == 'REVERSED', -1.0, 1.0)
active_txn['signed_qty'] = active_txn['qty_num'] * sign * rev_flip

hist_b = active_txn.groupby(['account_id', 'security_id'], as_index=False)['signed_qty'].sum()
hist_b = hist_b[hist_b['signed_qty'].abs() >= 1e-6]

book_vs_hist = pos.merge(hist_b, on=['account_id', 'security_id'], how='inner')
r03 = book_vs_hist[(book_vs_hist['qty_num'] - book_vs_hist['signed_qty']).abs() >= 0.01]
sign_inv = r03[(r03['qty_num'] + r03['signed_qty']).abs() < 0.01]
print(f"R03 book vs history breaks: {len(r03):,} (Report: 939) | Exact sign inversions: {len(sign_inv):,} (Report: 219)")

M02 domicile counts (bottom 10):
domicile
LU    12572
US    12556
CH    12491
IE    12487
JP    12429
SG    12400
CA    12386
GB    12379
ZZ      300
Name: count, dtype: int64

T06 100x price (qty>0): 1,117 (Report: 1,119)
T09 CUSIP mismatch (incl. blank txn CUSIP): 8,352 (Report: 12,598)
T07 orphan cancels (type=='CANCEL'): 0 (Report: 842)
R01 differing (tol >= 0.01): 1,917 (Report: 1,917)
R03 book vs history breaks: 939 (Report: 939) | Exact sign inversions: 219 (Report: 219)
